# AI Microdrama Generator (script layer, API-agnostic backend)

Turns a story prompt, optional character reference images, a genre template and optional required scenes into a shot-by-shot microdrama episode script, then hands each shot to a pluggable video backend.

## Architecture
`Brief` (prompt, characters + reference images, template, duration, language, scenes) -> `storyteller` (template + craft principles + beat timing prompt, JSON output, validate-and-repair loop up to 3 attempts) -> `Script` (shots with beat, visual, camera, speaker, dialogue, emotion) -> `VideoBackend.generate_shot` per shot using that shot's character references -> `assemble`.

Backends: `DryRunBackend` records exactly what would be sent. A real video API backend plugs in by subclassing `VideoBackend`. No video is generated in this version.

## Templates
hidden_identity, revenge, betrayal, ceo_romance, supernatural, mafia, family_inheritance. Each defines premise, engine, hook, twist and cliffhanger patterns.

## Structure rules
Hook inside 3s, setup, escalation, a twist at 60 to 85 percent, cliffhanger last. Shots 2 to 8s, at most 2.6 spoken words per second, max 3 to 4 main characters.

## Character reference handling
Each `Character` carries `ref_image` and a text description. Every shot lists which characters appear, and the backend receives only those references so identity stays consistent. Voice descriptions are stored per character for the audio stage.

## Benchmark
Six fixed briefs (7 templates, 3 languages, 60s and 90s) run per version. Per case: structural auto-score, LLM-judge rubric (8 criteria, temperature 0), and a human CSV including the main question "would I enjoy it". `compare` reports per-case and mean deltas between versions. Video rubric (voice, character, art style consistency, sync, watchability) is added once a video backend exists.

## Free-use note
Use any free OpenAI-compatible endpoint (for example NVIDIA build.nvidia.com free tier). The key is read with getpass and never stored.

In [ ]:
import os, sys, json, getpass
os.makedirs('microdrama', exist_ok=True)
sys.path.insert(0, os.getcwd())

### module `schema`

In [ ]:
%%writefile microdrama/schema.py
from dataclasses import dataclass, field, asdict
from typing import List, Optional
import json

@dataclass
class Character:
    name: str
    description: str = ""
    ref_image: Optional[str] = None
    voice: str = ""

@dataclass
class Brief:
    prompt: str
    characters: List[Character] = field(default_factory=list)
    template: str = "hidden_identity"
    duration_s: int = 60
    language: str = "en"
    style: str = "cinematic vertical 9:16, glossy shortform drama look"
    scenes: List[str] = field(default_factory=list)
    episode_number: int = 1

@dataclass
class Shot:
    idx: int
    start_s: float
    dur_s: float
    beat: str
    visual: str
    camera: str
    characters: List[str]
    speaker: Optional[str]
    dialogue: str
    emotion: str
    sfx_music: str = ""

@dataclass
class Script:
    title: str
    logline: str
    cliffhanger: str
    next_episode_teaser: str
    shots: List[Shot]

    def to_json(self):
        return json.dumps(asdict(self), ensure_ascii=False, indent=2)

    @staticmethod
    def from_dict(d):
        shots = [Shot(**s) for s in d["shots"]]
        return Script(d["title"], d["logline"], d["cliffhanger"], d.get("next_episode_teaser", ""), shots)


### module `templates`

In [ ]:
%%writefile microdrama/templates.py
TEMPLATES = {
    "hidden_identity": {
        "name": "Hidden identity",
        "premise": "A lead everyone underestimates secretly holds enormous power, wealth or status.",
        "engine": "dramatic irony: the audience knows the secret, the cast does not",
        "hook_patterns": ["public humiliation of the lead in the first 2 seconds", "a stranger bows to the lead, ignored by the bullies"],
        "twist_patterns": ["the villain's boss walks in and defers to the lead", "a phone call exposes the lead's true title"],
        "cliffhanger_patterns": ["the lead's secret is almost revealed, then cut at the reveal", "a black car arrives and someone says the lead's real name"],
    },
    "revenge": {
        "name": "Revenge",
        "premise": "A wronged lead returns with a plan to take apart the people who destroyed them.",
        "engine": "slow-burn payoff: every scene lands one precise blow",
        "hook_patterns": ["the lead smiles while being fired or betrayed", "a past photo is burned in close-up"],
        "twist_patterns": ["the villain's ally was the lead's plant all along", "the evidence is already in the lead's hands"],
        "cliffhanger_patterns": ["a recording starts playing in the middle of the villain's speech", "the lead says: this was only the first name on the list"],
    },
    "betrayal": {
        "name": "Betrayal and affair",
        "premise": "A trusted partner or friend is caught in a lie that changes everything.",
        "engine": "gossip: secrets, evidence, public exposure",
        "hook_patterns": ["a message notification lights up the wrong phone", "the lead overhears a whispered confession"],
        "twist_patterns": ["the other person is someone close to the lead", "the lover is the lead's boss"],
        "cliffhanger_patterns": ["a door opens and the third person walks in", "the partner says: I never loved you, I needed your name"],
    },
    "ceo_romance": {
        "name": "Contract marriage / CEO romance",
        "premise": "A forced or fake arrangement between opposites turns into real feeling under pressure.",
        "engine": "tension and almost-moments with status conflict",
        "hook_patterns": ["the cold CEO says: sign it or lose everything", "a stranger is mistaken for the fiance"],
        "twist_patterns": ["the CEO has been protecting the lead secretly", "the rival reveals a clause nobody read"],
        "cliffhanger_patterns": ["almost-kiss interrupted by a shocking call", "the CEO's ex appears claiming the same ring"],
    },
    "supernatural": {
        "name": "Werewolf / vampire / supernatural",
        "premise": "An ordinary human is pulled into a hidden bloodline war.",
        "engine": "forbidden bond plus danger and reveals of rank",
        "hook_patterns": ["glowing eyes in a mirror", "the lead is claimed in public by a feared alpha"],
        "twist_patterns": ["the lead's bloodline outranks everyone", "the trusted ally is the traitor"],
        "cliffhanger_patterns": ["the moon rises and the lead's body changes", "the council doors open and every head bows"],
    },
    "mafia": {
        "name": "Mafia / crime family",
        "premise": "Loyalty and love collide inside a ruthless family business.",
        "engine": "threat, loyalty tests, sudden violence implied not shown",
        "hook_patterns": ["a gun slides across a dinner table", "the lead is told: you know too much"],
        "twist_patterns": ["the informant is the don's own child", "the lead was the target of the hit all along"],
        "cliffhanger_patterns": ["lights go out and a single gunshot", "the don says: choose who dies"],
    },
    "family_inheritance": {
        "name": "Family and inheritance",
        "premise": "A reading of a will or family gathering explodes old resentments and secrets.",
        "engine": "ensemble gossip, shifting alliances, class conflict",
        "hook_patterns": ["the will names the least expected heir", "a lookalike child appears at the funeral"],
        "twist_patterns": ["the will has a hidden second page", "the heir was swapped at birth"],
        "cliffhanger_patterns": ["a DNA result is read aloud and cut", "the lawyer says: there is one more beneficiary, standing in this room"],
    },
}

STRUCTURE = {
    "hook_window_s": 3,
    "beats": [
        ("hook", 0.00, 0.08),
        ("setup", 0.08, 0.25),
        ("escalation", 0.25, 0.60),
        ("twist", 0.60, 0.85),
        ("cliffhanger", 0.85, 1.00),
    ],
    "max_words_per_second": 2.6,
    "min_shot_s": 2.0,
    "max_shot_s": 8.0,
    "max_main_characters": 4,
}

PRINCIPLES = [
    "Open on conflict or a visual question inside the first 3 seconds. No scene setting, no greetings.",
    "One clear emotional target per episode: the audience should want to see someone punished or rescued.",
    "Every shot changes something: new information, a power shift, or an escalation.",
    "Dialogue is short, blunt and quotable. Max 14 words per line. Subtext through reaction shots.",
    "Include one reversal at about 60 to 85 percent of the runtime that flips who holds power.",
    "End on an unresolved reveal or threat, cut mid-moment, with a specific open question.",
    "Use close-ups on faces for emotion and a held beat before each reveal.",
    "Make the villain easy to hate and the lead easy to root for within 10 seconds.",
    "Plant one detail early (object, line, name) that pays off at the twist.",
]


### module `llm`

In [ ]:
%%writefile microdrama/llm.py
import json, os, urllib.request, re

class LLM:
    def complete(self, system, user, temperature=0.8):
        raise NotImplementedError

class OpenAICompatLLM(LLM):
    def __init__(self, base_url, model, api_key=None, timeout=120):
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.api_key = api_key or os.environ.get("LLM_API_KEY", "")
        self.timeout = timeout

    def complete(self, system, user, temperature=0.8):
        body = json.dumps({"model": self.model, "temperature": temperature,
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}).encode()
        req = urllib.request.Request(self.base_url + "/chat/completions", data=body,
            headers={"Content-Type": "application/json", "Authorization": "Bearer " + self.api_key})
        with urllib.request.urlopen(req, timeout=self.timeout) as r:
            return json.loads(r.read())["choices"][0]["message"]["content"]

def extract_json(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b < 0:
        raise ValueError("no json object in model output")
    return json.loads(text[a:b + 1])


### module `validate`

In [ ]:
%%writefile microdrama/validate.py
from .templates import STRUCTURE

def validate_script(script, brief):
    issues = []
    names = {c.name for c in brief.characters}
    shots = script.shots
    if not shots:
        return ["no shots"]
    if abs(shots[0].start_s) > 0.01:
        issues.append("first shot must start at 0")
    t = 0.0
    for s in shots:
        if abs(s.start_s - t) > 0.25:
            issues.append(f"shot {s.idx} not contiguous (starts {s.start_s}, expected {round(t,1)})")
        if not STRUCTURE["min_shot_s"] - 0.01 <= s.dur_s <= STRUCTURE["max_shot_s"] + 0.01:
            issues.append(f"shot {s.idx} duration {s.dur_s}s outside limits")
        words = len(s.dialogue.split()) if s.dialogue else 0
        if s.dur_s > 0 and words / s.dur_s > STRUCTURE["max_words_per_second"] + 0.01:
            issues.append(f"shot {s.idx} dialogue too dense ({words} words in {s.dur_s}s)")
        if names:
            bad = [c for c in s.characters if c not in names]
            if bad:
                issues.append(f"shot {s.idx} uses unknown characters {bad}")
            if s.speaker and s.speaker not in names:
                issues.append(f"shot {s.idx} unknown speaker {s.speaker}")
        t = s.start_s + s.dur_s
    if abs(t - brief.duration_s) > 1.0:
        issues.append(f"total runtime {round(t,1)}s differs from target {brief.duration_s}s")
    if shots[0].beat != "hook" or shots[0].start_s > STRUCTURE["hook_window_s"]:
        issues.append("episode must open with a hook shot")
    if shots[-1].beat != "cliffhanger":
        issues.append("last shot must be the cliffhanger")
    beats = {s.beat for s in shots}
    for b in ("setup", "escalation", "twist"):
        if b not in beats:
            issues.append(f"missing beat: {b}")
    if not script.cliffhanger.strip():
        issues.append("cliffhanger field empty")
    for s in shots:
        if len(s.visual.split()) < 8:
            issues.append(f"shot {s.idx} visual too vague")
    return issues


### module `storyteller`

In [ ]:
%%writefile microdrama/storyteller.py
import json
from .schema import Script, Brief
from .templates import TEMPLATES, STRUCTURE, PRINCIPLES
from .llm import extract_json
from .validate import validate_script

SYSTEM = ("You are a head writer for top-performing vertical microdramas. You write tight shot-by-shot scripts "
          "that are addictive: instant hook, escalating conflict, a reversal, and a cliffhanger. "
          "Output only one JSON object, no prose.")

def beat_plan(duration_s):
    return [(n, round(a * duration_s, 1), round(b * duration_s, 1)) for n, a, b in STRUCTURE["beats"]]

def build_prompt(brief: Brief):
    t = TEMPLATES[brief.template]
    chars = "\n".join(f"- {c.name}: {c.description} (voice: {c.voice or 'natural'}; has reference image: {bool(c.ref_image)})" for c in brief.characters) or "- invent 2 to 4 characters"
    plan = "\n".join(f"- {n}: {a}s to {b}s" for n, a, b in beat_plan(brief.duration_s))
    scenes = "\n".join(f"- {s}" for s in brief.scenes) or "- none required"
    principles = "\n".join(f"{i+1}. {p}" for i, p in enumerate(PRINCIPLES))
    return f"""Write episode {brief.episode_number} of a microdrama.
Genre template: {t['name']}. Premise: {t['premise']}. Engine: {t['engine']}.
Hook ideas: {t['hook_patterns']}. Twist ideas: {t['twist_patterns']}. Cliffhanger ideas: {t['cliffhanger_patterns']}.
User story prompt: {brief.prompt}
Language of all dialogue: {brief.language}. Visual style: {brief.style}.
Total duration: {brief.duration_s} seconds. Shot length between {STRUCTURE['min_shot_s']} and {STRUCTURE['max_shot_s']} seconds. Max {STRUCTURE['max_words_per_second']} spoken words per second.
Characters (use only these names in the characters and speaker fields):
{chars}
Scenes the user requires:
{scenes}
Beat timing:
{plan}
Craft principles:
{principles}
JSON schema: {{"title": str, "logline": str, "cliffhanger": str, "next_episode_teaser": str,
"shots": [{{"idx": int, "start_s": float, "dur_s": float, "beat": "hook|setup|escalation|twist|cliffhanger",
"visual": str (concrete, filmable, includes setting, lighting, action), "camera": str, "characters": [names],
"speaker": name or null, "dialogue": str, "emotion": str, "sfx_music": str}}]}}
Shots must be contiguous, start at 0 and end at {brief.duration_s}."""

def write_script(llm, brief: Brief, max_attempts=3, log=print):
    prompt = build_prompt(brief)
    feedback = ""
    best = None
    for attempt in range(1, max_attempts + 1):
        raw = llm.complete(SYSTEM, prompt + feedback)
        try:
            script = Script.from_dict(extract_json(raw))
        except Exception as e:
            feedback = f"\nYour previous output was invalid ({e}). Return valid JSON only."
            log(f"attempt {attempt}: parse failure")
            continue
        issues = validate_script(script, brief)
        log(f"attempt {attempt}: {len(issues)} issues")
        if best is None or len(issues) < best[1]:
            best = (script, len(issues), issues)
        if not issues:
            return script, []
        feedback = "\nFix these problems and return the full corrected JSON:\n" + "\n".join("- " + i for i in issues)
    if best is None:
        raise RuntimeError("no valid script produced")
    return best[0], best[2]


### module `backend`

In [ ]:
%%writefile microdrama/backend.py
import os, json

class VideoBackend:
    name = "abstract"
    def generate_shot(self, shot, brief, out_dir):
        raise NotImplementedError
    def assemble(self, shot_paths, out_path):
        raise NotImplementedError

class DryRunBackend(VideoBackend):
    name = "dry_run"
    def generate_shot(self, shot, brief, out_dir):
        os.makedirs(out_dir, exist_ok=True)
        p = os.path.join(out_dir, f"shot_{shot.idx:02d}.json")
        refs = {c.name: c.ref_image for c in brief.characters if c.name in shot.characters}
        json.dump({"visual": shot.visual, "camera": shot.camera, "dur_s": shot.dur_s, "refs": refs,
                   "dialogue": shot.dialogue, "speaker": shot.speaker}, open(p, "w"), indent=1)
        return p
    def assemble(self, shot_paths, out_path):
        json.dump(shot_paths, open(out_path, "w"), indent=1)
        return out_path


### module `pipeline`

In [ ]:
%%writefile microdrama/pipeline.py
import os
from .storyteller import write_script
from .backend import DryRunBackend

def run_episode(llm, brief, out_dir, backend=None, log=print):
    os.makedirs(out_dir, exist_ok=True)
    script, issues = write_script(llm, brief, log=log)
    open(os.path.join(out_dir, "script.json"), "w").write(script.to_json())
    backend = backend or DryRunBackend()
    paths = [backend.generate_shot(s, brief, os.path.join(out_dir, "shots")) for s in script.shots]
    final = backend.assemble(paths, os.path.join(out_dir, "episode.out"))
    return {"script": script, "issues": issues, "output": final}


### module `benchmark`

In [ ]:
%%writefile microdrama/benchmark.py
import json, os, csv, time, statistics
from .schema import Brief, Character
from .pipeline import run_episode
from .validate import validate_script
from .llm import extract_json

CASES = [
    Brief("A delivery rider is mocked at a luxury gala; he is secretly the owner of the company that runs the building.", [Character("Arjun", "30s rider in a worn jacket"), Character("Vikram", "arrogant hotel heir"), Character("Meera", "sharp-eyed event manager")], "hidden_identity", 60, "en"),
    Brief("A fired accountant returns as the auditor of the firm that framed her.", [Character("Nina", "40s accountant, calm"), Character("Mr. Rao", "smug CFO"), Character("Dev", "junior analyst, secret ally")], "revenge", 60, "en"),
    Brief("A bride finds her husband's message to another woman minutes before the wedding vows.", [Character("Kavya", "bride in red"), Character("Rohan", "groom"), Character("Sana", "bridesmaid")], "betrayal", 60, "hi"),
    Brief("An intern is forced into a contract marriage with her ruthless boss to save her father's shop.", [Character("Lena", "intern"), Character("Mr. Kane", "cold CEO"), Character("Sofia", "Kane's ex")], "ceo_romance", 60, "en"),
    Brief("A waitress is claimed in public by a feared alpha during a full-moon feast.", [Character("Ivy", "waitress"), Character("Kael", "alpha"), Character("Brann", "beta rival")], "supernatural", 60, "en"),
    Brief("A will reading reveals a hidden heir sitting in the room.", [Character("Isabel", "eldest daughter"), Character("Tomas", "family lawyer"), Character("Luca", "stranger in a plain suit")], "family_inheritance", 90, "es"),
]

RUBRIC = {
    "hook": "Is there instant conflict or a strong visual question in the first 3 seconds?",
    "emotional_target": "Do you clearly want someone punished or rescued?",
    "escalation": "Does every shot change the situation?",
    "twist": "Is there a reversal that flips power and feels earned by a planted detail?",
    "cliffhanger": "Does it end on a specific open question that makes you want episode 2?",
    "dialogue": "Is the dialogue short, blunt, quotable, with no filler?",
    "juice": "Does it have gossip, intrigue and drama like real top microdramas?",
    "visual_filmability": "Are shots concrete and producible with consistent characters and look?",
}

VIDEO_RUBRIC = {
    "voice_consistency": "Does each character keep the same voice across shots?",
    "character_consistency": "Do faces, outfits and age stay the same across shots?",
    "art_style_consistency": "Is the look and colour grade stable across the episode?",
    "audio_video_sync": "Do lips, sound and action line up?",
    "watchability": "Would you keep watching, relative to real microdramas and shorts?",
}

def auto_score(script, brief):
    issues = validate_script(script, brief)
    shots = script.shots
    words = [len(s.dialogue.split()) for s in shots if s.dialogue]
    speakers = {s.speaker for s in shots if s.speaker}
    return {
        "structural_issues": len(issues),
        "shot_count": len(shots),
        "avg_shot_s": round(statistics.mean(s.dur_s for s in shots), 2) if shots else 0,
        "avg_line_words": round(statistics.mean(words), 1) if words else 0,
        "max_line_words": max(words) if words else 0,
        "speaker_count": len(speakers),
        "has_cliffhanger_beat": int(bool(shots) and shots[-1].beat == "cliffhanger"),
        "hook_in_3s": int(bool(shots) and shots[0].beat == "hook" and shots[0].start_s <= 3),
    }

JUDGE_SYSTEM = "You are a strict viewer who watches hundreds of microdramas. Score honestly; most scripts deserve 4 to 6. Output only JSON."

def judge(llm, script, brief):
    rubric = "\n".join(f"- {k}: {v}" for k, v in RUBRIC.items())
    user = f"""Score this microdrama script 1 to 10 on each criterion.
{rubric}
Return {{"scores": {{criterion: int}}, "weakest": str, "fix": str}}.
SCRIPT:
{script.to_json()}"""
    d = extract_json(llm.complete(JUDGE_SYSTEM, user, temperature=0.0))
    d["mean"] = round(statistics.mean(d["scores"].values()), 2)
    return d

def run_benchmark(gen_llm, judge_llm, out_root, version, backend=None, cases=None, log=print):
    cases = cases or CASES
    out = os.path.join(out_root, version)
    os.makedirs(out, exist_ok=True)
    rows = []
    for i, b in enumerate(cases):
        t0 = time.time()
        r = run_episode(gen_llm, b, os.path.join(out, f"case_{i:02d}"), backend, log=log)
        row = {"case": i, "template": b.template, "language": b.language, "duration_s": b.duration_s,
               "gen_seconds": round(time.time() - t0, 1)}
        row.update(auto_score(r["script"], b))
        try:
            j = judge(judge_llm, r["script"], b)
            row["judge_mean"] = j["mean"]
            row["judge_scores"] = j["scores"]
            row["judge_weakest"] = j.get("weakest", "")
        except Exception as e:
            row["judge_mean"] = None
            row["judge_error"] = str(e)
        rows.append(row)
        log(f"case {i} {b.template}: judge={row['judge_mean']} issues={row['structural_issues']}")
    json.dump(rows, open(os.path.join(out, "results.json"), "w"), indent=1)
    write_human_sheet(out, cases)
    return rows

def write_human_sheet(out, cases, include_video=False):
    crit = dict(RUBRIC)
    if include_video:
        crit.update(VIDEO_RUBRIC)
    with open(os.path.join(out, "human_scoring.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["case", "template"] + list(crit) + ["would_i_enjoy_it_1_10", "notes"])
        for i, b in enumerate(cases):
            w.writerow([i, b.template] + [""] * (len(crit) + 2))

def compare(out_root, v_old, v_new):
    a = {r["case"]: r for r in json.load(open(os.path.join(out_root, v_old, "results.json")))}
    b = {r["case"]: r for r in json.load(open(os.path.join(out_root, v_new, "results.json")))}
    lines = []
    for k in sorted(a):
        if k in b and a[k].get("judge_mean") is not None and b[k].get("judge_mean") is not None:
            lines.append((k, a[k]["judge_mean"], b[k]["judge_mean"], round(b[k]["judge_mean"] - a[k]["judge_mean"], 2)))
    avg = round(statistics.mean(x[3] for x in lines), 2) if lines else None
    return {"per_case": lines, "mean_delta": avg}


### module `__init__`

In [ ]:
%%writefile microdrama/__init__.py
from .schema import Brief, Character, Script, Shot
from .pipeline import run_episode


## Configure the LLM

In [ ]:
from microdrama.llm import OpenAICompatLLM
llm = OpenAICompatLLM('https://integrate.api.nvidia.com/v1', 'meta/llama-3.3-70b-instruct', getpass.getpass('LLM API key: '))

## Generate one episode

In [ ]:
from microdrama import Brief, Character, run_episode
brief = Brief('A delivery rider is mocked at a luxury gala; he is secretly the owner of the building.', [Character('Arjun','30s rider in a worn jacket'), Character('Vikram','arrogant hotel heir')], 'hidden_identity', 60)
result = run_episode(llm, brief, 'out/demo')
print(result['script'].title, '|', result['script'].cliffhanger)
print(result['issues'])

## Run the benchmark and compare versions

In [ ]:
from microdrama.benchmark import run_benchmark, compare
rows = run_benchmark(llm, llm, 'bench', 'v1')
for r in rows: print(r['case'], r['template'], r['judge_mean'], r['structural_issues'])

In [ ]:
rows2 = run_benchmark(llm, llm, 'bench', 'v2')
compare('bench', 'v1', 'v2')